# 第2章：大语言模型基础

本章深入探讨大语言模型（LLM）的核心技术原理，从 Transformer 架构出发，逐步理解自注意力机制、位置编码、预训练与微调等关键概念。同时介绍主流 LLM API（OpenAI、Anthropic Claude）的调用方式，以及如何根据任务需求选择合适的模型。

**核心知识点**：
- Transformer 架构与自注意力机制
- 位置编码原理
- Tokenization 分词机制
- OpenAI API 与 Claude API 调用
- 模型选择策略
- 上下文窗口与推理参数

## 学习目标与环境准备

**学习目标**：
1. 理解 Transformer 自注意力机制的计算原理
2. 掌握主流 LLM API 的调用方式
3. 能够根据任务需求选择合适的模型和参数

**环境准备**：本章主要依赖 Python 标准库和 `numpy`。API 调用部分可安装 `openai` 和 `anthropic` 库（可选）。

## 2.1 Transformer 自注意力机制实现

自注意力（Self-Attention）是 Transformer 的核心。每个位置的词通过 Query、Key、Value 三个矩阵计算与其他所有位置的注意力权重，从而捕捉全局上下文依赖关系。下面从头实现 Scaled Dot-Product Attention。

In [ ]:
import numpy as np


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(
    Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask: np.ndarray = None
) -> np.ndarray:
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    attention_weights = softmax(scores, axis=-1)
    output = np.matmul(attention_weights, V)
    return output, attention_weights


batch_size, num_heads, seq_len, d_k = 2, 1, 6, 4
np.random.seed(42)
Q = np.random.randn(batch_size, num_heads, seq_len, d_k)
K = np.random.randn(batch_size, num_heads, seq_len, d_k)
V = np.random.randn(batch_size, num_heads, seq_len, d_k)

causal_mask = np.triu(np.ones((seq_len, seq_len)) * -1e9, k=1)
causal_mask = causal_mask[np.newaxis, np.newaxis, :, :]

output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print(f"输入形状: Q={Q.shape}, K={K.shape}, V={V.shape}")
print(f"注意力权重形状: {attn_weights.shape}")
print(f"输出形状: {output.shape}")
print(f"第一个batch的注意力权重矩阵:
{np.round(attn_weights[0, 0], 3)}")
print(f"注意: 上三角为0（因果遮罩），每行和为1")

## 2.2 多头注意力机制

多头注意力（Multi-Head Attention）将 Q、K、V 投影到多个不同的子空间，使模型能够从不同角度关注信息。最终将所有头的输出拼接起来，通过线性变换得到最终结果。

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model: int = 512, num_heads: int = 8):
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.scale = np.sqrt(self.d_k)

        bound = np.sqrt(1.0 / d_model)
        self.W_q = np.random.uniform(-bound, bound, (d_model, d_model))
        self.W_k = np.random.uniform(-bound, bound, (d_model, d_model))
        self.W_v = np.random.uniform(-bound, bound, (d_model, d_model))
        self.W_o = np.random.uniform(-bound, bound, (d_model, d_model))

    def _split_heads(self, x: np.ndarray) -> np.ndarray:
        batch_size, seq_len, _ = x.shape
        x = x.reshape(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)

    def _merge_heads(self, x: np.ndarray) -> np.ndarray:
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(0, 2, 1, 3)
        return x.reshape(batch_size, seq_len, self.d_model)

    def forward(self, x: np.ndarray, mask: np.ndarray = None) -> np.ndarray:
        Q = np.matmul(x, self.W_q)
        K = np.matmul(x, self.W_k)
        V = np.matmul(x, self.W_v)

        Q = self._split_heads(Q)
        K = self._split_heads(K)
        V = self._split_heads(V)

        scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / self.scale
        if mask is not None:
            scores = scores + mask
        attn_weights = softmax(scores, axis=-1)
        attn_output = np.matmul(attn_weights, V)

        merged = self._merge_heads(attn_output)
        output = np.matmul(merged, self.W_o)
        return output, attn_weights


d_model, num_heads = 512, 8
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

x = np.random.randn(2, 10, d_model)
output, attn = mha.forward(x)

print(f"多头注意力参数: d_model={d_model}, num_heads={num_heads}, d_k={mha.d_k}")
print(f"输入形状: {x.shape}")
print(f"注意力权重形状: {attn.shape}")
print(f"输出形状: {output.shape}")
print(f"W_q 参数形状: {mha.W_q.shape}")
print(f"总参数量: {4 * d_model * d_model:,}")

## 2.3 位置编码

Transformer 本身不具备序列顺序感知能力，需要通过位置编码给每个位置注入位置信息。下面实现两种经典的位置编码：正弦位置编码和可学习位置编码。

In [ ]:
def sinusoidal_position_encoding(seq_len: int, d_model: int) -> np.ndarray:
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(
        np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model)
    )
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe


class LearnedPositionEncoding:
    def __init__(self, max_len: int, d_model: int):
        bound = np.sqrt(1.0 / d_model)
        self.embedding = np.random.uniform(-bound, bound, (max_len, d_model))

    def forward(self, positions: np.ndarray) -> np.ndarray:
        return self.embedding[positions]


seq_len, d_model = 20, 64
sin_pe = sinusoidal_position_encoding(seq_len, d_model)
learned_pe = LearnedPositionEncoding(max_len=100, d_model=d_model)

print(f"正弦位置编码形状: {sin_pe.shape}")
print(f"位置0的编码前8维: {np.round(sin_pe[0, :8], 4)}")
print(f"位置10的编码前8维: {np.round(sin_pe[10, :8], 4)}")

positions = np.arange(5)
learned_output = learned_pe.forward(positions)
print(f"可学习位置编码 (前5个位置, 前4维):")
print(np.round(learned_output[:, :4], 4))

## 2.4 简化版 BPE Tokenizer

LLM 处理文本前首先需要将文本转换为 token 序列。下面实现一个简化的 BPE（Byte Pair Encoding）分词器，帮助理解 tokenization 的基本原理。

In [ ]:
from collections import Counter
import re


class SimpleTokenizer:
    def __init__(self):
        self.vocab: dict = {}
        self.reverse_vocab: dict = {}
        self.merges: list = []

    def train(self, texts: list, vocab_size: int = 256):
        chars = set()
        for text in texts:
            chars.update(text)
        self.vocab = {i: ch for i, ch in enumerate(sorted(chars))}
        self.reverse_vocab = {ch: i for i, ch in self.vocab.items()}

        words = [list(t) for t in texts]
        while len(self.vocab) < vocab_size:
            pairs = Counter()
            for word in words:
                for i in range(len(word) - 1):
                    pairs[(word[i], word[i + 1])] += 1
            if not pairs:
                break
            best_pair = pairs.most_common(1)[0][0]
            new_token = ''.join(best_pair)
            new_id = len(self.vocab)
            self.vocab[new_id] = new_token
            self.reverse_vocab[new_token] = new_id
            self.merges.append(best_pair)

            new_words = []
            for word in words:
                new_word = []
                i = 0
                while i < len(word):
                    if i + 1 < len(word) and word[i] == best_pair[0] and word[i + 1] == best_pair[1]:
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_words.append(new_word)
            words = new_words

    def encode(self, text: str) -> list:
        tokens = list(text)
        for a, b in self.merges:
            merged_token = a + b
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i + 1 < len(tokens) and tokens[i] == a and tokens[i + 1] == b:
                    new_tokens.append(merged_token)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        return [self.reverse_vocab.get(t, 0) for t in tokens]

    def decode(self, ids: list) -> str:
        return ''.join(self.vocab.get(i, '?') for i in ids)


corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog play",
    "machine learning is fascinating",
]

tokenizer = SimpleTokenizer()
tokenizer.train(corpus, vocab_size=50)

print(f"词汇表大小: {len(tokenizer.vocab)}")
print(f"合并操作数: {len(tokenizer.merges)}")
print(f"前10个merge: {tokenizer.merges[:10]}")

test_text = "the cat sat"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"原文: '{test_text}'")
print(f"编码: {encoded}")
print(f"解码: '{decoded}'")

## 2.5 Transformer 编码器层

完整的 Transformer 编码器层由多头自注意力、前馈神经网络（FFN）、层归一化和残差连接组成。下面实现一个标准的 Transformer Encoder Layer。

In [ ]:
def layer_norm(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    mean = np.mean(x, axis=-1, keepdims=True)
    std = np.std(x, axis=-1, keepdims=True)
    return (x - mean) / (std + eps)


class FeedForward:
    def __init__(self, d_model: int = 512, d_ff: int = 2048):
        bound1 = np.sqrt(1.0 / d_model)
        bound2 = np.sqrt(1.0 / d_ff)
        self.W1 = np.random.uniform(-bound1, bound1, (d_model, d_ff))
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.uniform(-bound2, bound2, (d_ff, d_model))
        self.b2 = np.zeros(d_model)

    def forward(self, x: np.ndarray) -> np.ndarray:
        hidden = np.maximum(0, np.matmul(x, self.W1) + self.b1)
        return np.matmul(hidden, self.W2) + self.b2


class TransformerEncoderLayer:
    def __init__(self, d_model: int = 512, num_heads: int = 8, d_ff: int = 2048):
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x: np.ndarray, mask: np.ndarray = None) -> np.ndarray:
        attn_out, _ = self.attention.forward(x, mask)
        x = layer_norm(x + attn_out)

        ffn_out = self.ffn.forward(x)
        x = layer_norm(x + ffn_out)
        return x


encoder_layer = TransformerEncoderLayer(d_model=512, num_heads=8, d_ff=2048)
input_seq = np.random.randn(2, 10, 512)
output = encoder_layer.forward(input_seq)

print(f"Transformer 编码器层")
print(f"输入形状: {input_seq.shape}")
print(f"输出形状: {output.shape}")
print(f"输出均值: {output.mean():.4f}, 标准差: {output.std():.4f}")

num_params = (512 * 512 * 4) + (512 * 2048 + 2048 * 512)
print(f"编码器层参数量: ~{num_params:,}")

## 2.6 OpenAI API 调用封装

在实际开发中，大多数开发者通过 API 调用 LLM。下面封装一个统一的 OpenAI API 调用类，支持 Chat Completions、流式输出和系统提示词。

In [ ]:
import os
import json
from typing import Optional, List, Dict


class OpenAIClient:
    def __init__(self, api_key: Optional[str] = None, base_url: Optional[str] = None):
        self.api_key = api_key or os.environ.get("OPENAI_API_KEY", "sk-placeholder")
        self.base_url = base_url or "https://api.openai.com/v1"
        self.models = {
            "gpt-4o": {"max_tokens": 128000, "cost_input": 2.50, "cost_output": 10.00},
            "gpt-4o-mini": {"max_tokens": 128000, "cost_input": 0.15, "cost_output": 0.60},
            "gpt-3.5-turbo": {"max_tokens": 16385, "cost_input": 0.50, "cost_output": 1.50},
            "o3-mini": {"max_tokens": 200000, "cost_input": 1.10, "cost_output": 4.40},
        }

    def chat(
        self,
        messages: List[Dict[str, str]],
        model: str = "gpt-4o-mini",
        temperature: float = 0.7,
        max_tokens: int = 1024,
        stream: bool = False,
    ) -> Dict:
        payload = {
            "model": model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "stream": stream,
        }
        return {
            "id": "chatcmpl-mock-001",
            "model": model,
            "choices": [{
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": f"[模拟响应] 收到 {len(messages)} 条消息，模型 {model}",
                },
                "finish_reason": "stop",
            }],
            "usage": {
                "prompt_tokens": 50,
                "completion_tokens": 100,
                "total_tokens": 150,
            },
        }

    def simple_ask(self, prompt: str, system: str = "", model: str = "gpt-4o-mini") -> str:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        response = self.chat(messages, model=model)
        return response["choices"][0]["message"]["content"]

    def estimate_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        info = self.models.get(model, {})
        cost = (
            input_tokens * info.get("cost_input", 0) / 1_000_000
            + output_tokens * info.get("cost_output", 0) / 1_000_000
        )
        return cost


client = OpenAIClient()

print("=== 基础对话调用 ===")
result = client.chat([{"role": "user", "content": "什么是Transformer?"}])
print(f"模型: {result['model']}")
print(f"回复: {result['choices'][0]['message']['content']}")
print(f"Token用量: {result['usage']}")

print(f"=== 带系统提示词的调用 ===")
response = client.simple_ask(
    "总结一下AI Agent的特点",
    system="你是一位专业的AI技术讲师，回答简洁明了。",
)
print(f"回复: {response}")

print(f"=== 费用估算 ===")
cost = client.estimate_cost("gpt-4o", 1000, 500)
print(f"gpt-4o 费用估算: ${cost:.6f} (1000输入 + 500输出)")
cost_mini = client.estimate_cost("gpt-4o-mini", 1000, 500)
print(f"gpt-4o-mini 费用估算: ${cost_mini:.6f} (1000输入 + 500输出)")

## 2.7 Anthropic Claude API 调用封装

除了 OpenAI，Anthropic 的 Claude 系列模型也是 Agent 开发的主流选择。下面封装 Claude API 调用，展示 Messages API 的使用方式。

In [ ]:
class ClaudeClient:
    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key or os.environ.get("ANTHROPIC_API_KEY", "sk-ant-placeholder")
        self.models = {
            "claude-sonnet-4-20250514": {
                "max_tokens": 200000,
                "cost_input": 3.00,
                "cost_output": 15.00,
            },
            "claude-3-5-haiku-20241022": {
                "max_tokens": 200000,
                "cost_input": 0.80,
                "cost_output": 4.00,
            },
            "claude-opus-4-20250514": {
                "max_tokens": 200000,
                "cost_input": 15.00,
                "cost_output": 75.00,
            },
        }

    def messages_create(
        self,
        messages: List[Dict[str, str]],
        model: str = "claude-sonnet-4-20250514",
        system: str = "",
        max_tokens: int = 1024,
        temperature: float = 0.7,
    ) -> Dict:
        payload = {
            "model": model,
            "system": system,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature,
        }
        return {
            "id": "msg_mock_001",
            "model": model,
            "type": "message",
            "role": "assistant",
            "content": [{
                "type": "text",
                "text": f"[Claude模拟] 基于 {len(messages)} 条消息的回复",
            }],
            "stop_reason": "end_turn",
            "usage": {
                "input_tokens": 60,
                "output_tokens": 90,
            },
        }

    def simple_ask(self, prompt: str, system: str = "", model: str = "claude-sonnet-4-20250514") -> str:
        messages = [{"role": "user", "content": prompt}]
        response = self.messages_create(messages, model=model, system=system)
        return response["content"][0]["text"]


claude = ClaudeClient()

print("=== Claude 基础对话 ===")
result = claude.messages_create(
    [{"role": "user", "content": "解释一下ReAct循环"}],
    system="你是一个AI技术专家",
)
print(f"模型: {result['model']}")
print(f"回复: {result['content'][0]['text']}")
print(f"停止原因: {result['stop_reason']}")

print(f"=== Token 用量对比 ===")
print(f"OpenAI用量格式: {{prompt_tokens, completion_tokens, total_tokens}}")
print(f"Claude用量格式: {{input_tokens, output_tokens}}")
print(f"Claude Token用量: {result['usage']}")

## 2.8 模型选择器

不同任务需要不同的模型。下面实现一个智能模型选择器，根据任务类型、预算限制和性能要求自动推荐最合适的模型。

In [ ]:
from dataclasses import dataclass
from enum import Enum


class TaskType(Enum):
    SIMPLE_CHAT = "simple_chat"
    CODE_GENERATION = "code_generation"
    ANALYSIS = "analysis"
    CREATIVE_WRITING = "creative_writing"
    LONG_CONTEXT = "long_context"
    MULTI_STEP = "multi_step"


@dataclass
class ModelInfo:
    name: str
    provider: str
    context_window: int
    cost_per_1m_input: float
    cost_per_1m_output: float
    strengths: List[str]


class ModelSelector:
    def __init__(self):
        self.models = [
            ModelInfo("gpt-4o", "openai", 128000, 2.50, 10.00,
                      ["code_generation", "analysis", "multi_step"]),
            ModelInfo("gpt-4o-mini", "openai", 128000, 0.15, 0.60,
                      ["simple_chat", "code_generation"]),
            ModelInfo("claude-sonnet-4-20250514", "anthropic", 200000, 3.00, 15.00,
                      ["code_generation", "analysis", "long_context", "creative_writing"]),
            ModelInfo("claude-3-5-haiku-20241022", "anthropic", 200000, 0.80, 4.00,
                      ["simple_chat", "code_generation"]),
            ModelInfo("claude-opus-4-20250514", "anthropic", 200000, 15.00, 75.00,
                      ["analysis", "multi_step", "long_context", "creative_writing"]),
            ModelInfo("o3-mini", "openai", 200000, 1.10, 4.40,
                      ["code_generation", "analysis", "multi_step"]),
        ]

    def recommend(
        self,
        task_type: TaskType,
        max_budget: float = float("inf"),
        need_long_context: bool = False,
    ) -> List[ModelInfo]:
        candidates = []
        for model in self.models:
            score = 0
            if task_type.value in model.strengths:
                score += 3
            if need_long_context and model.context_window >= 128000:
                score += 2
            avg_cost = (model.cost_per_1m_input + model.cost_per_1m_output) / 2
            if avg_cost <= max_budget / 1000:
                score += 2
            candidates.append((model, score))
        candidates.sort(key=lambda x: -x[1])
        return [m for m, s in candidates if s > 0][:3]

    def compare_models(self, task_type: TaskType, input_tokens: int = 1000, output_tokens: int = 500):
        recommendations = self.recommend(task_type)
        print(f"任务类型: {task_type.value}")
        print(f"{'模型':<35} {'提供商':<12} {'费用($)':<10} {'上下文':<10}")
        print("-" * 70)
        for m in recommendations:
            cost = (m.cost_per_1m_input * input_tokens + m.cost_per_1m_output * output_tokens) / 1_000_000
            print(f"{m.name:<35} {m.provider:<12} ${cost:<9.6f} {m.context_window//1000}K")


selector = ModelSelector()

for task in [TaskType.CODE_GENERATION, TaskType.ANALYSIS, TaskType.SIMPLE_CHAT, TaskType.CREATIVE_WRITING]:
    selector.compare_models(task)

print(f"=== 带预算限制的选择 ===")
cheap = selector.recommend(TaskType.SIMPLE_CHAT, max_budget=1.0)
print(f"预算限制 $1/百万token → 推荐: {[m.name for m in cheap]}")
expensive = selector.recommend(TaskType.ANALYSIS, max_budget=100.0)
print(f"预算限制 $100/百万token → 推荐: {[m.name for m in expensive]}")

## 练习

1. **自注意力可视化**：修改 `scaled_dot_product_attention` 函数，为注意力权重矩阵生成热力图可视化（可使用 matplotlib）。

2. **扩展 BPE Tokenizer**：为 `SimpleTokenizer` 添加特殊 token（如 `<|endoftext|>`、`<|user|>` 等）的处理逻辑，并实现子词正则化。

3. **API 流式输出**：扩展 `OpenAIClient`，实现 `stream=True` 时的流式输出处理，逐 token 打印响应内容。

4. **模型选择策略优化**：修改 `ModelSelector`，引入延迟指标（假设不同模型的平均响应时间），在费用和速度之间做权衡推荐。